In [1]:
# =========================
# Mount Google Drive (Colab)
# =========================
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# ============================================================
# Synthetic Clinical Pharmacology Prompt Generator (JSONL) — FIXED
# Key fix:
#   - STOP forcing each NEW_MED into all 3 buckets (this created fake "safe" cases)
#   - Use 3 NEW_MED pools: SAFE / BORDERLINE / DANGEROUS
#   - Guarantee coverage only within each pool's intended bucket
#   - Make SAFE archetypes truly "obvious safe" (no red flags, minimal polypharmacy)
# ============================================================

import json
import random
import hashlib
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional, Tuple

# ----------------------------
# CONFIG
# ----------------------------
SEED = 42
N_CASES = 10000
OUT_PATH = "clinical_pharm_prompts_10000.jsonl"

RISK_MIX = {
    "safe": 0.40,
    "borderline": 0.30,
    "dangerous": 0.30,
}

# Coverage per NEW_MED within its OWN bucket pool
MIN_PER_BUCKET = 120  # safe pool + borderline pool + dangerous pool should fit under N_CASES

random.seed(SEED)

PROMPT_TEMPLATE = """QUESTION:
You are a clinical pharmacology decision assistant.
Always base your answers on verified medical knowledge, and avoid speculation.

Patient profile:
- Age: {age}
- Sex: {sex}
- Weight: {weight_kg} kg
- Height: {height_cm} cm
- BMI: {bmi:.1f}
- Renal function: {renal_function}
- Liver function: {liver_function}
- Medical history: {medical_history}
- Current medications: {current_meds}
- New medication considered: {new_med}

Tasks:
1) Decide if the drug combination is safe for this patient. Answer only "Safe" or "Unsafe".
2) Provide a single explanation (4-8 sentences) that summarizes the mechanism of risk or compatibility, including:
   * pharmacokinetic and/or pharmacodynamic interactions
   * effect of comorbidities (especially kidney or liver function)
   * mechanisms that lead to toxicity or adverse outcomes
3) If there is uncertainty in the reasoning, state how a physician should resolve the uncertainty safely.

Format the output exactly as:

Decision: <Safe/Unsafe>
Explanation: <4-8 sentences>
Uncertainty notes: <optional>
"""

# ----------------------------
# Helpers: patient anthropometrics
# ----------------------------
def sample_sex() -> str:
    return random.choice(["Male", "Female"])

def sample_age_safe() -> int:
    # younger / mid to make truly "obvious safe" more frequent
    return int(round(min(75, max(18, random.gauss(mu=42, sigma=12)))))

def sample_age_other() -> int:
    # broader, slightly older skew
    return int(round(min(90, max(18, random.gauss(mu=58, sigma=16)))))

def sample_height_cm(sex: str) -> int:
    if sex == "Male":
        h = random.gauss(175, 8)
    else:
        h = random.gauss(162, 7)
    return int(round(min(200, max(140, h))))

def sample_weight_kg(height_cm: int) -> int:
    bmi = min(45.0, max(17.0, random.gauss(26.0, 6.0)))
    h_m = height_cm / 100.0
    w = bmi * (h_m ** 2)
    return int(round(min(160, max(45, w))))

def bmi_from(height_cm: int, weight_kg: int) -> float:
    h_m = height_cm / 100.0
    return weight_kg / (h_m ** 2)

RENAL_BUCKETS = [
    "Normal (eGFR ≥ 90)",
    "Mild impairment (eGFR 60–89)",
    "Moderate impairment (eGFR 30–59)",
    "Severe impairment (eGFR < 30)",
    "Dialysis",
]

LIVER_BUCKETS = [
    "No known impairment",
    "Chronic liver disease (compensated)",
    "Cirrhosis (compensated / Child-Pugh A)",
    "Cirrhosis (moderate / Child-Pugh B)",
    "Decompensated cirrhosis (Child-Pugh C)",
]

HISTORY_COMMON = [
    "Hypertension",
    "Type 2 diabetes mellitus",
    "Coronary artery disease",
    "Atrial fibrillation",
    "Heart failure (HFrEF)",
    "Heart failure (HFpEF)",
    "COPD",
    "Asthma",
    "Chronic kidney disease",
    "Chronic liver disease",
    "Cirrhosis",
    "Decompensated cirrhosis",
    "History of GI bleeding",
    "Peptic ulcer disease",
    "Osteoarthritis",
    "Rheumatoid arthritis",
    "Depression",
    "Generalized anxiety disorder",
    "Epilepsy",
    "Bipolar disorder",
    "Schizophrenia",
    "Hypothyroidism",
    "Hyperlipidemia",
    "Gout",
    "Long QT syndrome",
    "Hypokalemia (recurrent)",
]

MEDS = {
    # (keep your existing pool, we only add a few clean meds)
    "warfarin": {"tags": ["bleeding"], "class": "anticoagulant"},
    "apixaban": {"tags": ["bleeding"], "class": "DOAC"},
    "rivaroxaban": {"tags": ["bleeding"], "class": "DOAC"},
    "dabigatran": {"tags": ["bleeding"], "class": "DOAC"},
    "aspirin": {"tags": ["bleeding", "gi_irritant"], "class": "antiplatelet"},
    "clopidogrel": {"tags": ["bleeding"], "class": "antiplatelet"},

    "valproic acid": {"tags": ["hepatotoxic", "enzyme_inhib"], "class": "antiepileptic"},
    "carbamazepine": {"tags": ["enzyme_inducer"], "class": "antiepileptic"},
    "phenytoin": {"tags": ["enzyme_inducer"], "class": "antiepileptic"},
    "lamotrigine": {"tags": ["rash_risk"], "class": "antiepileptic"},
    "levetiracetam": {"tags": [], "class": "antiepileptic"},

    "clozapine": {"tags": ["myelosuppress"], "class": "antipsychotic"},
    "quetiapine": {"tags": ["qt"], "class": "antipsychotic"},
    "haloperidol": {"tags": ["qt"], "class": "antipsychotic"},
    "sertraline": {"tags": ["serotonergic"], "class": "SSRI"},
    "fluoxetine": {"tags": ["serotonergic", "enzyme_inhib"], "class": "SSRI"},
    "tranylcypromine": {"tags": ["maoi"], "class": "MAOI"},

    "trimethoprim-sulfamethoxazole": {"tags": ["myelosuppress", "warfarin_interact"], "class": "antibiotic"},
    "ciprofloxacin": {"tags": ["qt", "warfarin_interact"], "class": "antibiotic"},
    "clarithromycin": {"tags": ["qt", "enzyme_inhib"], "class": "antibiotic"},
    "fluconazole": {"tags": ["enzyme_inhib", "warfarin_interact"], "class": "antifungal"},

    "lisinopril": {"tags": ["hyperkalemia"], "class": "ACEi"},
    "losartan": {"tags": ["hyperkalemia"], "class": "ARB"},
    "spironolactone": {"tags": ["hyperkalemia"], "class": "K_sparing_diuretic"},
    "furosemide": {"tags": [], "class": "loop_diuretic"},
    "hydrochlorothiazide": {"tags": ["lithium_interact"], "class": "thiazide"},
    "amiodarone": {"tags": ["qt", "warfarin_interact"], "class": "antiarrhythmic"},
    "metoprolol": {"tags": [], "class": "beta_blocker"},
    "atorvastatin": {"tags": [], "class": "statin"},

    "metformin": {"tags": ["renal_caution"], "class": "biguanide"},
    "lithium": {"tags": ["narrow_ti"], "class": "mood_stabilizer"},
    "ibuprofen": {"tags": ["renal_risk", "gi_irritant", "bleeding"], "class": "NSAID"},
    "naproxen": {"tags": ["renal_risk", "gi_irritant", "bleeding"], "class": "NSAID"},
    "prednisone": {"tags": ["immunosuppress"], "class": "steroid"},
    "methotrexate": {"tags": ["hepatotoxic", "myelosuppress"], "class": "DMARD"},
    "allopurinol": {"tags": [], "class": "xanthine_oxidase_inhibitor"},

    "sildenafil": {"tags": ["nitrate_interact"], "class": "PDE5i"},
    "isosorbide mononitrate": {"tags": ["nitrate"], "class": "nitrate"},

    "digoxin": {"tags": ["narrow_ti"], "class": "cardiac_glycoside"},
    "sotalol": {"tags": ["qt"], "class": "antiarrhythmic"},
    "ondansetron": {"tags": ["qt"], "class": "antiemetic"},
    "rifampin": {"tags": ["enzyme_inducer"], "class": "antibiotic"},
    "linezolid": {"tags": ["serotonergic", "myelosuppress"], "class": "antibiotic"},
    "erythromycin": {"tags": ["qt", "enzyme_inhib"], "class": "antibiotic"},
    "valacyclovir": {"tags": [], "class": "antiviral"},

    # NEW: "obvious safe" meds for current-med filler if needed
    "paracetamol": {"tags": [], "class": "analgesic"},
    "loratadine": {"tags": [], "class": "antihistamine"},
    "amoxicillin": {"tags": [], "class": "antibiotic"},
}

# ----------------------------
# NEW MED POOLS (this is the key fix)
# ----------------------------
NEW_MED_SAFE_POOL = [
    "paracetamol (500 mg every 6 hours as needed, max 3 g/day)",
    "loratadine (10 mg once daily)",
    "amoxicillin (500 mg three times daily)",
    "valacyclovir (1000 mg twice daily)",
    "atorvastatin (20 mg once daily)",
    "prednisone (20 mg once daily for 5 days)",
]

NEW_MED_BORDERLINE_POOL = [
    "metformin (1000 mg twice daily)",
    "ibuprofen (600 mg three times daily as needed)",
    "naproxen (500 mg twice daily)",
    "lithium (300 mg twice daily)",
    "fluconazole (200 mg once daily)",
    "hydrochlorothiazide (25 mg once daily)",
]

NEW_MED_DANGEROUS_POOL = [
    "methotrexate (15 mg once weekly)",
    "trimethoprim-sulfamethoxazole (800/160 mg twice daily)",
    "clarithromycin (500 mg twice daily)",
    "ciprofloxacin (500 mg twice daily)",
    "linezolid (600 mg twice daily)",
    "erythromycin (500 mg four times daily)",
    "rifampin (600 mg once daily)",
    "amiodarone (200 mg once daily)",
    "sotalol (80 mg twice daily)",
    "ondansetron (8 mg three times daily as needed)",
    "spironolactone (25 mg once daily)",
    "sildenafil (50 mg as needed)",
]

def sample_history(min_n=1, max_n=4) -> List[str]:
    n = random.randint(min_n, max_n)
    return random.sample(HISTORY_COMMON, k=n)

# ----------------------------
# Archetypes
# ----------------------------
def scenario_dangerous(forced_new_med: Optional[str] = None) -> Tuple[Dict, List[str], str]:
    archetypes = []
    archetypes.append((
        {"liver_function": "Decompensated cirrhosis (Child-Pugh C)", "history_add": ["Decompensated cirrhosis"]},
        ["valproic acid"], "methotrexate (15 mg once weekly)"
    ))
    archetypes.append((
        {"history_add": ["Atrial fibrillation"], "renal_function": "Normal (eGFR ≥ 90)"},
        ["warfarin", "metoprolol"], "trimethoprim-sulfamethoxazole (800/160 mg twice daily)"
    ))
    archetypes.append((
        {"history_add": ["Depression"]},
        ["tranylcypromine"], "sertraline (50 mg once daily)"
    ))
    archetypes.append((
        {"history_add": ["Coronary artery disease"]},
        ["isosorbide mononitrate", "metoprolol"], "sildenafil (50 mg as needed)"
    ))
    archetypes.append((
        {"history_add": ["Long QT syndrome", "Hypokalemia (recurrent)"]},
        ["quetiapine"], "clarithromycin (500 mg twice daily)"
    ))

    overrides, cur, newm_default = random.choice(archetypes)
    newm = forced_new_med if forced_new_med is not None else newm_default
    return overrides, list(dict.fromkeys(cur)), newm

def scenario_borderline(forced_new_med: Optional[str] = None) -> Tuple[Dict, List[str], str]:
    archetypes = []
    archetypes.append((
        {"renal_function": "Moderate impairment (eGFR 30–59)", "history_add": ["Type 2 diabetes mellitus", "Chronic kidney disease"]},
        ["atorvastatin"], "metformin (1000 mg twice daily)"
    ))
    archetypes.append((
        {"renal_function": random.choice(["Mild impairment (eGFR 60–89)", "Moderate impairment (eGFR 30–59)"]), "history_add": ["Hypertension"]},
        ["lisinopril", "furosemide"], random.choice(["ibuprofen (600 mg three times daily as needed)", "naproxen (500 mg twice daily)"])
    ))
    archetypes.append((
        {"history_add": ["Bipolar disorder", "Hypertension"], "renal_function": random.choice(["Normal (eGFR ≥ 90)", "Mild impairment (eGFR 60–89)"])},
        ["lithium"], "hydrochlorothiazide (25 mg once daily)"
    ))
    archetypes.append((
        {"history_add": ["Atrial fibrillation"]},
        ["warfarin"], "fluconazole (200 mg once daily)"
    ))

    overrides, cur, newm_default = random.choice(archetypes)
    newm = forced_new_med if forced_new_med is not None else newm_default
    return overrides, list(dict.fromkeys(cur)), newm

def scenario_safe(forced_new_med: Optional[str] = None) -> Tuple[Dict, List[str], str]:
    # Clean safe: normal organs, minimal meds, no anticoagulants, no QT flags, no MAOI, etc.
    archetypes = []
    archetypes.append((
        {"renal_function": "Normal (eGFR ≥ 90)", "liver_function": "No known impairment", "history_add": ["Osteoarthritis"]},
        ["paracetamol"], "paracetamol (500 mg every 6 hours as needed, max 3 g/day)"
    ))
    archetypes.append((
        {"renal_function": "Normal (eGFR ≥ 90)", "liver_function": "No known impairment", "history_add": ["Generalized anxiety disorder"]},
        ["loratadine"], "loratadine (10 mg once daily)"
    ))
    archetypes.append((
        {"renal_function": "Normal (eGFR ≥ 90)", "liver_function": "No known impairment", "history_add": ["Asthma"]},
        ["albuterol"] if False else ["loratadine"], "amoxicillin (500 mg three times daily)"
    ))
    archetypes.append((
        {"renal_function": "Normal (eGFR ≥ 90)", "liver_function": "No known impairment", "history_add": ["Hyperlipidemia"]},
        ["atorvastatin"], "valacyclovir (1000 mg twice daily)"
    ))

    overrides, cur, newm_default = random.choice(archetypes)
    newm = forced_new_med if forced_new_med is not None else newm_default
    return overrides, list(dict.fromkeys(cur)), newm

# ----------------------------
# Case dataclass
# ----------------------------
@dataclass
class Case:
    id: str
    risk_bucket: str
    age: int
    sex: str
    weight_kg: int
    height_cm: int
    bmi: float
    renal_function: str
    liver_function: str
    medical_history: List[str]
    current_meds: List[str]
    new_med: str
    prompt: str

def stable_case_id(payload: dict) -> str:
    raw = json.dumps(payload, sort_keys=True).encode("utf-8")
    return hashlib.sha1(raw).hexdigest()[:12]

def choose_risk_bucket() -> str:
    r = random.random()
    cum = 0.0
    for k, p in RISK_MIX.items():
        cum += p
        if r <= cum:
            return k
    return "safe"

def build_case(risk_bucket: str, forced_new_med: Optional[str] = None) -> Case:
    sex = sample_sex()

    if risk_bucket == "safe":
        age = sample_age_safe()
        renal_function = "Normal (eGFR ≥ 90)"
        liver_function = "No known impairment"
    else:
        age = sample_age_other()
        renal_function = random.choice(RENAL_BUCKETS[:4])
        liver_function = random.choice(LIVER_BUCKETS[:4])

    height_cm = sample_height_cm(sex)
    weight_kg = sample_weight_kg(height_cm)
    bmi = bmi_from(height_cm, weight_kg)

    history = sample_history()

    if risk_bucket == "dangerous":
        overrides, forced_current, newm_default = scenario_dangerous(forced_new_med)
        new_med = forced_new_med if forced_new_med is not None else newm_default
    elif risk_bucket == "borderline":
        overrides, forced_current, newm_default = scenario_borderline(forced_new_med)
        new_med = forced_new_med if forced_new_med is not None else newm_default
    else:
        overrides, forced_current, newm_default = scenario_safe(forced_new_med)
        new_med = forced_new_med if forced_new_med is not None else newm_default

    # Apply overrides
    renal_function = overrides.get("renal_function", renal_function)
    liver_function = overrides.get("liver_function", liver_function)
    history = list(dict.fromkeys(history + overrides.get("history_add", [])))

    # Current meds:
    current = forced_current[:]

    # IMPORTANT: extras policy
    if risk_bucket == "safe":
        # keep it clean: 0 extras (or at most 1 very benign)
        extra_n = random.choice([0, 0, 1])
        benign = ["paracetamol", "loratadine", "atorvastatin", "valacyclovir", "metoprolol"]
        extras = [m for m in benign if m not in current]
    else:
        extra_n = random.choice([0, 1, 2])
        extras = [m for m in MEDS.keys() if m not in current and m != "methotrexate"]

    if extras and extra_n > 0:
        current += random.sample(extras, k=min(extra_n, len(extras)))

    current = list(dict.fromkeys(current))

    # Light coherence
    if "Decompensated cirrhosis" in history:
        liver_function = "Decompensated cirrhosis (Child-Pugh C)"

    prompt = PROMPT_TEMPLATE.format(
        age=age,
        sex=sex,
        weight_kg=weight_kg,
        height_cm=height_cm,
        bmi=bmi,
        renal_function=renal_function,
        liver_function=liver_function,
        medical_history=", ".join(history),
        current_meds=", ".join(current),
        new_med=new_med
    )

    payload_for_id = {
        "risk_bucket": risk_bucket,
        "age": age, "sex": sex,
        "weight_kg": weight_kg, "height_cm": height_cm,
        "renal_function": renal_function,
        "liver_function": liver_function,
        "medical_history": history,
        "current_meds": current,
        "new_med": new_med,
    }
    cid = stable_case_id(payload_for_id)

    return Case(
        id=cid,
        risk_bucket=risk_bucket,
        age=age,
        sex=sex,
        weight_kg=weight_kg,
        height_cm=height_cm,
        bmi=bmi,
        renal_function=renal_function,
        liver_function=liver_function,
        medical_history=history,
        current_meds=current,
        new_med=new_med,
        prompt=prompt
    )

def generate_dataset(n: int) -> List[Case]:
    seen = set()
    cases: List[Case] = []

    # Coverage guarantee per pool (NOT across all 3 buckets)
    def add_coverage(pool: List[str], bucket: str):
        nonlocal cases, seen
        for new_med in pool:
            added = 0
            tries = 0
            while added < MIN_PER_BUCKET and tries < MIN_PER_BUCKET * 300:
                tries += 1
                c = build_case(bucket, forced_new_med=new_med)
                if c.id in seen:
                    continue
                seen.add(c.id)
                cases.append(c)
                added += 1
            if added < MIN_PER_BUCKET:
                print(f"Warning: coverage shortfall for ({bucket}, {new_med}) -> {added}/{MIN_PER_BUCKET}")

    required = MIN_PER_BUCKET * (len(NEW_MED_SAFE_POOL) + len(NEW_MED_BORDERLINE_POOL) + len(NEW_MED_DANGEROUS_POOL))
    if required > n:
        raise ValueError(f"MIN_PER_BUCKET too large: required {required} > N_CASES={n}")

    add_coverage(NEW_MED_SAFE_POOL, "safe")
    add_coverage(NEW_MED_BORDERLINE_POOL, "borderline")
    add_coverage(NEW_MED_DANGEROUS_POOL, "dangerous")

    # Fill the rest
    attempts = 0
    while len(cases) < n and attempts < n * 80:
        attempts += 1
        bucket = choose_risk_bucket()
        if bucket == "safe":
            new_med = random.choice(NEW_MED_SAFE_POOL)
        elif bucket == "borderline":
            new_med = random.choice(NEW_MED_BORDERLINE_POOL)
        else:
            new_med = random.choice(NEW_MED_DANGEROUS_POOL)

        c = build_case(bucket, forced_new_med=new_med)
        if c.id in seen:
            continue
        seen.add(c.id)
        cases.append(c)

    return cases[:n]

def main(out_path: str = OUT_PATH):
    cases = generate_dataset(N_CASES)

    with open(out_path, "w", encoding="utf-8") as f:
        for c in cases:
            f.write(json.dumps(asdict(c), ensure_ascii=False) + "\n")

    # quick summary
    bucket_counts = {"safe": 0, "borderline": 0, "dangerous": 0}
    for c in cases:
        bucket_counts[c.risk_bucket] += 1

    print("Wrote:", out_path)
    print("Bucket counts:", bucket_counts)
    print("Example prompt:\n")
    print(cases[0].prompt)
    return cases

cases = main()


Wrote: clinical_pharm_prompts_10000.jsonl
Bucket counts: {'safe': 3541, 'borderline': 2828, 'dangerous': 3631}
Example prompt:

QUESTION:
You are a clinical pharmacology decision assistant.
Always base your answers on verified medical knowledge, and avoid speculation.

Patient profile:
- Age: 52
- Sex: Male
- Weight: 86 kg
- Height: 176 cm
- BMI: 27.8
- Renal function: Normal (eGFR ≥ 90)
- Liver function: No known impairment
- Medical history: Epilepsy, Hyperlipidemia
- Current medications: atorvastatin
- New medication considered: paracetamol (500 mg every 6 hours as needed, max 3 g/day)

Tasks:
1) Decide if the drug combination is safe for this patient. Answer only "Safe" or "Unsafe".
2) Provide a single explanation (4-8 sentences) that summarizes the mechanism of risk or compatibility, including:
   * pharmacokinetic and/or pharmacodynamic interactions
   * effect of comorbidities (especially kidney or liver function)
   * mechanisms that lead to toxicity or adverse outcomes
3) If t

In [5]:
import os, json
from dataclasses import asdict

OUTPUT_DIR = "/content/drive/MyDrive/DL_Local"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUT_PATH = os.path.join(OUTPUT_DIR, "clinical_pharm_prompts_10000.jsonl")

with open(OUT_PATH, "w", encoding="utf-8") as f:
    for c in cases:
        f.write(json.dumps(asdict(c), ensure_ascii=False) + "\n")

print("Saved to:", OUT_PATH)


Saved to: /content/drive/MyDrive/DL_Local/clinical_pharm_prompts_10000.jsonl


In [ ]:
# ============================================================
# VERIFY GENERATED DATASET (JSONL)
# Run AFTER: cases = generate_dataset(N_CASES)
# ============================================================

import os, json
from collections import Counter, defaultdict

# If you saved to Drive, set this path accordingly
DRIVE_DIR = "/content/drive/MyDrive/clinical_kd_dataset"
OUT_PATH = os.path.join(DRIVE_DIR, "clinical_pharm_prompts_6000.jsonl")

# If you're not saving yet, you can verify the in-memory `cases` too.
# This code supports both:
#  - verifying from file (recommended)
#  - verifying from `cases` if file isn't present

REQUIRED_KEYS = {
    "id", "risk_bucket", "age", "sex", "weight_kg", "height_cm", "bmi",
    "renal_function", "liver_function", "medical_history", "current_meds",
    "new_med", "prompt"
}

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception as e:
                raise RuntimeError(f"JSON parse error on line {i}: {repr(e)}")
    return rows

def summarize_rows(rows, min_per_bucket=None):
    # Basic size
    print("Rows:", len(rows))

    # Key presence
    missing_any = 0
    for r in rows:
        if not REQUIRED_KEYS.issubset(r.keys()):
            missing_any += 1
    print("Rows missing required keys:", missing_any)

    # Buckets
    bucket_counts = Counter(r.get("risk_bucket") for r in rows)
    print("\nBucket counts:", dict(bucket_counts))

    # Duplicates
    ids = [r.get("id") for r in rows]
    dup_ids = len(ids) - len(set(ids))
    print("Duplicate ids:", dup_ids)

    # Cross-tab: new_med x bucket
    cross = defaultdict(lambda: Counter())
    new_med_counts = Counter()
    for r in rows:
        nm = r["new_med"]
        b = r["risk_bucket"]
        cross[nm][b] += 1
        new_med_counts[nm] += 1

    print("\n#Unique new_med:", len(new_med_counts))
    print("Top 5 new_med counts:", new_med_counts.most_common(5))
    print("Bottom 5 new_med counts:", sorted(new_med_counts.items(), key=lambda x: x[1])[:5])

    # Coverage guarantee
    if min_per_bucket is not None:
        worst = None
        worst_val = 10**9
        for nm, bc in cross.items():
            for b in ["safe", "borderline", "dangerous"]:
                v = bc.get(b, 0)
                if v < worst_val:
                    worst_val = v
                    worst = (nm, b, v)
        print(f"\nMin per (new_med,bucket): {worst_val}  | worst pair: {worst}")
        violations = []
        for nm, bc in cross.items():
            for b in ["safe", "borderline", "dangerous"]:
                if bc.get(b, 0) < min_per_bucket:
                    violations.append((nm, b, bc.get(b, 0)))
        print("Coverage violations:", len(violations))
        if violations:
            print("First 10 violations:", violations[:10])

    # Challenge proxies
    def has_history(r, needle):
        return any(needle.lower() in h.lower() for h in r.get("medical_history", []))

    severe_ckd = sum(("Severe impairment" in r["renal_function"] or "Dialysis" in r["renal_function"]) for r in rows)
    child_pugh_c = sum(("Child-Pugh C" in r["liver_function"]) for r in rows)
    long_qt = sum(has_history(r, "Long QT") for r in rows)
    hypok = sum(has_history(r, "Hypokalemia") for r in rows)

    avg_current_meds = sum(len(r.get("current_meds", [])) for r in rows) / max(1, len(rows))
    avg_history = sum(len(r.get("medical_history", [])) for r in rows) / max(1, len(rows))

    print("\nChallenge proxies:")
    print("Avg # current meds:", round(avg_current_meds, 2))
    print("Avg # history items:", round(avg_history, 2))
    print("Severe CKD/Dialysis count:", severe_ckd, f"({severe_ckd/len(rows):.1%})")
    print("Child-Pugh C count:", child_pugh_c, f"({child_pugh_c/len(rows):.1%})")
    print("Long QT history count:", long_qt, f"({long_qt/len(rows):.1%})")
    print("Hypokalemia history count:", hypok, f"({hypok/len(rows):.1%})")

    # Prompt sanity check
    bad_prompt = 0
    for r in rows[:200]:  # sample to keep it fast
        p = r.get("prompt", "")
        if "Decision:" not in p or "Explanation:" not in p or "Patient profile:" not in p:
            bad_prompt += 1
    print("\nPrompt sanity sample (first 200 rows): bad prompts =", bad_prompt)

# -----------------------------
# 1) Verify in-memory cases (if available)
# -----------------------------
rows_from_cases = None
if "cases" in globals():
    try:
        rows_from_cases = [asdict(c) for c in cases]
        print("Found in-memory `cases` ✅")
    except Exception as e:
        print("Found `cases` but couldn't convert via asdict:", repr(e))

# -----------------------------
# 2) Verify file (if present)
# -----------------------------
if os.path.exists(OUT_PATH):
    print("\nFound file on Drive ✅", OUT_PATH)
    rows = load_jsonl(OUT_PATH)
    summarize_rows(rows, min_per_bucket=globals().get("MIN_PER_BUCKET", None))
else:
    print("\nDrive file not found at:", OUT_PATH)
    if rows_from_cases is not None:
        print("Verifying in-memory `cases` instead.")
        summarize_rows(rows_from_cases, min_per_bucket=globals().get("MIN_PER_BUCKET", None))
    else:
        print("No `cases` in memory either. Run generation first.")


Found in-memory `cases` ✅

Found file on Drive ✅ /content/drive/MyDrive/clinical_kd_dataset/clinical_pharm_prompts_6000.jsonl
Rows: 6000
Rows missing required keys: 0

Bucket counts: {'safe': 2075, 'borderline': 1960, 'dangerous': 1965}
Duplicate ids: 0

#Unique new_med: 25
Top 5 new_med counts: [('amiodarone (200 mg once daily)', 259), ('fluoxetine (20 mg once daily)', 252), ('prednisone (20 mg once daily for 5 days)', 247), ('fluconazole (200 mg once daily)', 245), ('spironolactone (25 mg once daily)', 245)]
Bottom 5 new_med counts: [('digoxin (0.125 mg once daily)', 227), ('valacyclovir (1000 mg twice daily)', 228), ('methotrexate (15 mg once weekly)', 232), ('rifampin (600 mg once daily)', 233), ('isosorbide mononitrate (30 mg once daily)', 233)]

Min per (new_med,bucket): 70  | worst pair: ('ibuprofen (600 mg three times daily as needed)', 'borderline', 70)
Coverage violations: 0

Challenge proxies:
Avg # current meds: 2.44
Avg # history items: 3.89
Severe CKD/Dialysis count: 139 

In [ ]:
# ============================================================
# VERIFICATION CELL #2 — Leakage + "Borderline is Borderline"
# Run AFTER you already loaded `rows` from the JSONL file
# (i.e., rows is a list of dicts, length ~6000)
# ============================================================

import math
from collections import defaultdict, Counter

# ----------------------------
# Helpers
# ----------------------------
def entropy(counter: Counter) -> float:
    total = sum(counter.values())
    if total == 0:
        return 0.0
    h = 0.0
    for v in counter.values():
        if v <= 0:
            continue
        p = v / total
        h -= p * math.log2(p)
    return h

def normalize_text(s: str) -> str:
    s = s.lower()
    for ch in ["(", ")", ",", ".", ";", ":", "—", "-", "/", "\\", "[", "]", "{", "}", "\n", "\t"]:
        s = s.replace(ch, " ")
    while "  " in s:
        s = s.replace("  ", " ")
    return s.strip()

def bucket_from_row(r):
    b = r.get("risk_bucket")
    if b not in ("safe", "borderline", "dangerous"):
        return None
    return b

def has_history(r, needle: str) -> bool:
    needle = needle.lower()
    return any(needle in h.lower() for h in r.get("medical_history", []))

def renal_severe(r) -> int:
    rf = r.get("renal_function", "")
    return int(("Severe impairment" in rf) or ("Dialysis" in rf))

def liver_c(r) -> int:
    lf = r.get("liver_function", "")
    return int("Child-Pugh C" in lf)

# ----------------------------
# 0) Basic sanity about rows
# ----------------------------
assert isinstance(rows, list) and len(rows) > 0, "rows must be a non-empty list of dicts"

print("Rows:", len(rows))
print("Buckets:", Counter(bucket_from_row(r) for r in rows))

# ============================================================
# 1) Leakage / shortcut check:
#    Find tokens that strongly predict a bucket (low entropy)
# ============================================================

token_buckets = defaultdict(Counter)
token_total = Counter()

for r in rows:
    b = bucket_from_row(r)
    if b is None:
        continue

    # Use structured fields only (avoid prompt boilerplate)
    blob = " ".join([
        r.get("new_med", ""),
        " ".join(r.get("current_meds", [])),
        " ".join(r.get("medical_history", [])),
        r.get("renal_function", ""),
        r.get("liver_function", ""),
    ])
    blob = normalize_text(blob)
    toks = set(blob.split())  # set prevents overweighting repeats inside the same case

    for t in toks:
        token_buckets[t][b] += 1
        token_total[t] += 1

# compute suspicious tokens
SUSPICIOUS_MIN_COUNT = 120   # ignore rare tokens
ENTROPY_THRESHOLD = 0.65     # 0=perfectly predictive, ~1.585=max for 3 buckets

suspicious = []
for tok, c in token_buckets.items():
    total = token_total[tok]
    if total < SUSPICIOUS_MIN_COUNT:
        continue
    h = entropy(c)
    if h < ENTROPY_THRESHOLD:
        suspicious.append((tok, total, dict(c), round(h, 3)))

suspicious.sort(key=lambda x: (x[3], -x[1]))  # lowest entropy first, break ties by frequency

print("\n=== Leakage check: suspicious bucket-predictive tokens ===")
if not suspicious:
    print("No suspicious tokens found under current thresholds ✅")
else:
    print("Showing up to 30 most suspicious tokens:")
    for tok, total, counts, h in suspicious[:30]:
        print(f"tok='{tok}' total={total} counts={counts} entropy={h}")

print("\nNote: some tokens may be clinically meaningful (e.g., 'warfarin'),")
print("but if you see a weird generator artifact token here, that's a red flag.")

# ============================================================
# 2) "Borderline is borderline" check:
#    Compare structural difficulty indicators across buckets
# ============================================================

bucket_rows = {"safe": [], "borderline": [], "dangerous": []}
for r in rows:
    b = bucket_from_row(r)
    if b:
        bucket_rows[b].append(r)

def mean(vals):
    return sum(vals) / len(vals) if vals else float("nan")

def pct(vals):
    return 100.0 * mean(vals)

def summarize_bucket(name: str, rs: list):
    n = len(rs)
    meds = [len(r.get("current_meds", [])) for r in rs]
    hist = [len(r.get("medical_history", [])) for r in rs]

    sev_ckd = [renal_severe(r) for r in rs]
    c_pugh = [liver_c(r) for r in rs]
    long_qt = [int(has_history(r, "Long QT")) for r in rs]
    hypok = [int(has_history(r, "Hypokalemia")) for r in rs]
    afib = [int(has_history(r, "Atrial fibrillation")) for r in rs]

    print(f"\n=== {name.upper()} (n={n}) ===")
    print("Avg # current meds:", round(mean(meds), 2))
    print("Avg # history items:", round(mean(hist), 2))
    print("Severe CKD/Dialysis %:", round(pct(sev_ckd), 1))
    print("Child-Pugh C %:", round(pct(c_pugh), 1))
    print("Long QT history %:", round(pct(long_qt), 1))
    print("Hypokalemia history %:", round(pct(hypok), 1))
    print("Atrial fibrillation %:", round(pct(afib), 1))

print("\n=== Borderline-ness check (should look intermediate on many metrics) ===")
summarize_bucket("safe", bucket_rows["safe"])
summarize_bucket("borderline", bucket_rows["borderline"])
summarize_bucket("dangerous", bucket_rows["dangerous"])

# ============================================================
# 3) Quick prompt boilerplate sanity: ensure prompt starts with "QUESTION:" and contains required headings
# ============================================================

REQ_PROMPT_SNIPPETS = ["QUESTION:", "Patient profile:", "Tasks:", "Decision:", "Explanation:"]
bad = 0
sample_n = min(300, len(rows))
for r in rows[:sample_n]:
    p = r.get("prompt", "")
    if any(s not in p for s in REQ_PROMPT_SNIPPETS):
        bad += 1

print(f"\nPrompt structure check (first {sample_n} rows): bad={bad}")
if bad == 0:
    print("Prompt structure looks consistent ✅")
else:
    print("Some prompts are missing required sections ❌ (inspect those rows)")

# ============================================================
# Optional: Fail-fast assertions (uncomment if you want strictness)
# ============================================================
# assert bad == 0, "Prompt structure errors found"
# assert not suspicious[:10], "Found highly predictive leakage tokens; consider adjusting generator"


Rows: 6000
Buckets: Counter({'safe': 2075, 'dangerous': 1965, 'borderline': 1960})

=== Leakage check: suspicious bucket-predictive tokens ===
Showing up to 30 most suspicious tokens:
tok='<' total=139 counts={'dangerous': 139} entropy=0.0
tok='severe' total=139 counts={'dangerous': 139} entropy=0.0

Note: some tokens may be clinically meaningful (e.g., 'warfarin'),
but if you see a weird generator artifact token here, that's a red flag.

=== Borderline-ness check (should look intermediate on many metrics) ===

=== SAFE (n=2075) ===
Avg # current meds: 2.25
Avg # history items: 4.08
Severe CKD/Dialysis %: 0.0
Child-Pugh C %: 9.0
Long QT history %: 9.8
Hypokalemia history %: 9.0
Atrial fibrillation %: 31.1

=== BORDERLINE (n=1960) ===
Avg # current meds: 2.2
Avg # history items: 3.79
Severe CKD/Dialysis %: 0.0
Child-Pugh C %: 9.9
Long QT history %: 10.0
Hypokalemia history %: 9.7
Atrial fibrillation %: 27.7

=== DANGEROUS (n=1965) ===
Avg # current meds: 2.89
Avg # history items: 3.78
S